In [92]:
from itertools import combinations_with_replacement
from collections import namedtuple
import re
from functools import lru_cache

In [93]:
with open("../../data/2025/day12.txt") as f:
    data = f.read()

In [3]:
data = """0:
###
##.
##.

1:
###
##.
.##

2:
.##
###
##.

3:
##.
###
##.

4:
###
#..
###

5:
###
.#.
###

4x4: 0 0 0 0 2 0
12x5: 1 0 1 0 2 2
12x5: 1 0 1 0 3 2"""

In [94]:
Shape = namedtuple('Shape', 'w h coords')
Region = namedtuple('Region', 'w h items')

def shape_parse(shape_schema: list) -> Shape:
    w, h, shape_coords = len(shape_schema[0]), len(shape_schema), set()
    for tile_y, row in enumerate(shape_schema):
        for tile_x, tile in enumerate(row):
            if tile != '.': shape_coords.add(tile_x + tile_y * 1j)
    return Shape(w, h, frozenset(shape_coords))

def shape_rotate_90(shape: Shape, turns: int) -> Shape:
    center = shape.w // 2 + shape.h // 2 * 1j
    r = 1j ** turns
    return Shape(
        shape.w,
        shape.h,
        frozenset((xy-center) * r + center for xy in shape.coords)
    )

def shape_translate(shape: Shape, offset: complex) -> Shape:
    return Shape(
        shape.w,
        shape.h,
        frozenset(xy + offset for xy in shape.coords)
    )

def shape_flip(shape: Shape, axis: str="x"):
    center = shape.w // 2 + shape.h // 2 * 1j
    return Shape(
        shape.w,
        shape.h,
        frozenset((-1 if axis=='y' else 1) * (xy-center).conjugate() + center
                  for xy in shape.coords)
    )

def shape_transforms(shape: Shape):
    results = []
    for flip_shape in [shape, shape_flip(shape, 'x'), shape_flip(shape, 'y')]:
       for turn in range(4):
            new_shape = shape_rotate_90(flip_shape, turn)
            if new_shape not in results: results.append(new_shape)
    return results

In [95]:
sections = data.split("\n\n")
shape_schemas = [line.splitlines()[1:] for line in sections[:-1]]
regions = [Region(*(int(w), int(h), [int(i) for i in items.split(' ')]))
           for line in sections[-1].splitlines()
           for w,h,items in re.findall(r'(\d+)x(\d+): (.*)', line)]
shapes = [shape_parse(shape) for shape_id, shape in enumerate(shape_schemas)]
shape_transforms_cache = [shape_transforms(shape) for shape_id, shape in enumerate(shapes)]
shape_areas = {i: len(shape.coords) for i, shape in enumerate(shapes)}

In [96]:
def ifp(width, height, piece):
    max_x = int(max(c.real for c in piece))
    max_y = int(max(c.imag for c in piece))

    return {complex(x, y)
            for x in range(width - max_x)
            for y in range(height - max_y)}

@lru_cache(maxsize=None)
def remaining_area_needed(remaining_pieces: tuple[int]):
    return sum(shape_areas[p] for p in remaining_pieces)

def area_feasible(remaining_pieces, occupied, region):
    needed = remaining_area_needed(tuple(remaining_pieces))
    available = (region.w * region.h) - len(occupied)
    return needed <= available

def feasible(shape, shape_id, transform_id, occupied):
    forbidden = {oc - pc for oc in occupied for pc in shape.coords}
    return ifp_cache[(shape_id, transform_id)] - forbidden

def place(pos, shape: Shape, occupied: set):
    return occupied | {pos + c for c in shape.coords}

def unplace(pos, shape: Shape, occupied: set):
    return occupied - {pos + c for c in shape.coords}

In [97]:
def solve(remaining, occupied):
    if not remaining: return occupied  # All pieces placed

    if not area_feasible(remaining, occupied, region): return False

    best_id = None
    best_options = None

    for shape_id in list(dict.fromkeys(remaining)):
        options = []

        for transform_id, shape in enumerate(shape_transforms_cache[shape_id]):
            for at in feasible(shape, shape_id, transform_id, occupied):
                options.append((shape, at))

        if not options:
            return False  # Impossible, prune

        if best_options is None or len(options) < len(best_options):
            best_id = shape_id
            best_options = options

    for shape, at in best_options:
        new_remaining = remaining.copy()
        new_remaining.remove(best_id)
        if results := solve(new_remaining, place(at, shape, occupied)):
            return results

    return False  # No position worked, backtrack

In [89]:
ifp_cache = {}
part1 = 0

regions = [
    Region(w=6, h=4, items=[0,0,0,0,0,0])
]

for region in regions:
    #print(region)
    ifp_cache.clear()
    for shape_id, transforms in enumerate(shape_transforms_cache):
        for transform_id, transform in enumerate(transforms):
            ifp_cache[(shape_id, transform_id)] = ifp(region.w, region.h, transform.coords)

    combos = list(combinations_with_replacement([i for i,_ in enumerate(shapes)], 4))

    #shapes_to_pack = [x for sublist in [[shape_id]*quantity
    #  for shape_id, quantity in enumerate(region.items) if quantity > 0]
    #  for x in sublist]

    for combo in combos:
        shapes_to_pack = list(combo)
        occupied = set()

        if occupied := solve(shapes_to_pack, occupied):
            part1 += 1
            print(occupied)
            print("Packed!")
            print(shapes_to_pack)

            for grid_y in range(region.h):
                for grid_x in range(region.w):
                    print('#' if (grid_x + grid_y * 1j) in occupied else '.', end='')
                print('')
            print('')
        else:
            #print("Failed to pack")
            pass

print(f"Part 1: {part1}")

{0j, (1+0j), (3+0j), (1+1j), (2+1j), (4+0j), (4+1j), (2+2j), (5+1j), 1j, (5+2j), 2j, (1+2j), (1+3j), (2+3j), (3+2j), (4+2j), (4+3j), (5+3j), (3+1j)}
Packed!
[3, 3, 3, 3]
##.##.
######
######
.##.##

{0j, (1+0j), (3+0j), (1+1j), (2+1j), (4+0j), (4+1j), (2+2j), (5+1j), 1j, (5+2j), 2j, (1+2j), (1+3j), (2+3j), (3+2j), (4+2j), (3+3j), (4+3j), (5+3j), (3+1j)}
Packed!
[3, 3, 3, 4]
##.##.
######
######
.#####

{0j, (1+0j), (3+0j), (1+1j), (2+1j), (4+0j), (4+1j), (2+2j), (5+1j), 1j, (5+2j), 2j, (1+2j), 3j, (1+3j), (2+3j), (3+2j), (4+2j), (3+3j), (4+3j), (5+3j), (3+1j)}
Packed!
[3, 3, 4, 4]
##.##.
######
######
######

{0j, (1+0j), 1j, (1+1j), (2+1j), 2j, (1+2j), (2+2j), 3j, (1+3j), (2+3j), (3+1j), (3+2j), (4+2j), (3+3j), (4+3j), (5+3j), (3+0j), (5+2j), (4+0j), (5+0j), (4+1j), (5+1j)}
Packed!
[3, 4, 4, 4]
##.###
######
######
######

{0j, (1+0j), (2+0j), 1j, (1+1j), (2+1j), 2j, (1+2j), (3+1j), 3j, (1+3j), (2+3j), (3+2j), (4+2j), (3+3j), (4+3j), (5+3j), (2+2j), (3+0j), (5+2j), (4+0j), (5+0j), (4+

In [99]:
ifp_cache = {}
part1 = 0

for region in regions[2:3]:
    print(region)
    ifp_cache.clear()
    for shape_id, transforms in enumerate(shape_transforms_cache):
        for transform_id, transform in enumerate(transforms):
            ifp_cache[(shape_id, transform_id)] = ifp(region.w, region.h, transform.coords)

    shapes_to_pack = [x for sublist in [[shape_id]*quantity
     for shape_id, quantity in enumerate(region.items) if quantity > 0]
     for x in sublist]

    occupied = set()

    if occupied := solve(shapes_to_pack, occupied):
        part1 += 1
        print(occupied)
        print("Packed!")
        #print(shapes_to_pack)

        # for grid_y in range(region.h):
        #     for grid_x in range(region.w):
        #         print('#' if (grid_x + grid_y * 1j) in occupied else '.', end='')
        #     print('')
        # print('')
    else:
        print("Failed to pack")

print(f"Part 1: {part1}")

Region(w=43, h=45, items=[30, 41, 37, 40, 37, 25])


KeyboardInterrupt: 

In [70]:
from itertools import combinations

#combos = combinations([shape for groups in shape_transforms_cache for shape in groups], 2)
combos = combinations(shapes * 2, 2)
print(len(list(combos)))

66


In [56]:
shape_transforms_cache

[[Shape(w=3, h=3, coords=frozenset({(1+0j), (2+0j), 1j, (1+1j), 2j, (1+2j), (2+2j)})),
  Shape(w=3, h=3, coords=frozenset({0j, (1+0j), 1j, (1+1j), (2+1j), 2j, (2+2j)})),
  Shape(w=3, h=3, coords=frozenset({0j, (1+0j), (2+0j), (1+1j), (2+1j), 2j, (1+2j)})),
  Shape(w=3, h=3, coords=frozenset({0j, (2+0j), 1j, (1+1j), (2+1j), (1+2j), (2+2j)})),
  Shape(w=3, h=3, coords=frozenset({0j, (1+0j), (2+0j), 1j, (1+1j), (1+2j), (2+2j)})),
  Shape(w=3, h=3, coords=frozenset({(1+0j), (2+0j), 1j, (1+1j), (2+1j), 2j, (2+2j)})),
  Shape(w=3, h=3, coords=frozenset({0j, (1+0j), (1+1j), (2+1j), 2j, (1+2j), (2+2j)})),
  Shape(w=3, h=3, coords=frozenset({0j, (2+0j), 1j, (1+1j), (2+1j), 2j, (1+2j)}))],
 [Shape(w=3, h=3, coords=frozenset({0j, (1+0j), 1j, (1+1j), 2j, (1+2j), (2+2j)})),
  Shape(w=3, h=3, coords=frozenset({0j, (1+0j), (2+0j), 1j, (1+1j), (2+1j), 2j})),
  Shape(w=3, h=3, coords=frozenset({0j, (1+0j), (2+0j), (1+1j), (2+1j), (1+2j), (2+2j)})),
  Shape(w=3, h=3, coords=frozenset({(2+0j), 1j, (1+1j)